## Tutorial 
# Compare NEON and EMIT data for SOAP site
## First of two notebooks
### Authors: Randi Neff, Hannah Rieder and Bridget Hass
#### last updated: 7/29/25

This tutorial is intended for Earth Science data professionals. Additional details and an overall summary of this project is available on the [Earth Lab Blog](https://earthlab.colorado.edu/earth-data-analytics-professional-graduate-certificate/earth-data-analytics-certificate-cohorts). In this tutorial, we will learn how to evaluate forest health using a calculation of the Canopy Water Content (CWC) from individual tiles at the Soaproot Saddle (SOAP) field site in the Sierra National Forest in California. The hyperspectral data for the CWC calculation comes from the National Ecological Observatory Network's (NEON) Level 3 Spectrometer orthorectified surface directional reflectance - mosaic data product and the Earth Surface Mineral Dust Source Investigation (EMIT) L2A Reflectance Data Product. 

## The objectives of this tutorial (divided between two notebooks) are to:
* Use co-located data from NEON and EMIT
* Calculate Canopy Water Content (CWC) from NEON and EMIT hyperspectral data
* Evaluate CWC data at different scales
* Compare between burned and unburned areas

DATA
The data provided with this tutorial were derived from existing code at:
* NEON Spectrometer orthorectified surface bidirectional reflectance data.
* Shapefiles for Creek fire boundary and NEON burned and unburned tiles which are found in the DATA folder.
* EMIT L2A Estimated Surface Reflectance granule(s) that cover the NEON burned and unburned tiles.
* [Land Processes Distributed Active Archive Center (LP DAAC)](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html#cwc-of-a-single-point).

Additional data will be downloaded programmatically within this tutorial.

## Tutorial Outline for Notebook 1 - NEON and EMIT Data 
* You will need a NEON user account, but will be provided with shapefiles for burned/unburned tile boundaries 
* NEON tile boundaries will be used to crop EMIT data to the same region of interest (ROI)
* You will need a NASA Earthdata account and functions found in the script folder

## Tutorial Outline for Notebook 2 - Canopy Water Content Comparison
* Open NEON and EMIT Reflectance Data
* Calculate Canopy Water Content (CWC)
* Compare CWC Datasets

## Notes on functions that are used in both notebooks and found in the scripts folder
* The two datasets (NEON & EMIT) are very large and in different formats so the aop_h5refl2xarray function does conversions to make them compatible
* The calc_ewt function was originally developed for EMIT data and some metadata is hardcoded so preserved while maintaining functionality
* The surfrfl_hvplot_image assists with visualizing both datasets

Additional considerations are discussed in the [README file](https://github.com/NEONScience/AOP-EMIT/blob/main/README.md)

## 0 Imports

In [ ]:
# Import packages
import os, sys # management of files and directories
import pathlib
# Some cells may generate warnings that we can ignore.
# Comment below lines to see.
import warnings
warnings.filterwarnings('ignore')

import xarray as xr # working with multi-dimensional arrays
import rasterio as rio # raster library
import hvplot.xarray # graphing
import pandas as pd # data manipulation with dataframes and 1D arrays  
import geopandas as gpd # add geometry to panda dataframes

import requests # downloading data from online sources
import earthaccess # accessing NASA Earth data
import folium # visualizing geospatial data
import math

import csv # working with tabular data
import holoviews as hv # interactive visualizations
import netCDF4 as nc # read and write NetCDF files

from zipfile import ZipFile # handling data that comes in zipped formats
from branca.element import Figure # structuring the HTML output
from IPython.display import display # working in Jupyter notebooks
from rasterio.plot import show, show_hist # Generates and displays a histogram 
# of the raster data

import neonutilities as nu #work with NEON reflectance data
import h5py #work with NEON reflectance data
import numpy as np
import matplotlib.pyplot as plt

## 1. Setup

In the setup section, we will:
1. create directories to store the data and functions for this project,
2. download and import the extra necessary scripts needed for this tutorial, and
3. download shapefiles to crop EMIT data to the NEON region of interest (ROI)

### 1.1 Create Data and Scripts (modules) Directories

The directories we will make are: an overarching data directory, a reflectance data directory, a CWC data directory, and a modules directory.

The overarching data directory will contain the reflectance and CWC data directories and a CSV file containing lab measurements of the complex refractive index of liquid water. 

The reflectance data directory should contain the two EMIT cropped datasets (one for the burned tile and one for the unburned tile) and the two NEON reflectance datasets (one for the burned tile and one for the unburned tile) created in Tutorial Notebook 01. All of these datasets should be NetCDF files.

The CWC data directory will be where we store the results of this tutorial notebook: the CWC calculations of the burned and unburned tiles calculated using the cropped EMIT and NEON reflectance data.

The modules directory must be in the same folder as where you have these tutorial notebooks stored for some of the imported functions to work. In this modules directory, we will manually download some Python scripts (.py files). The scripts contain various functions we'll use in this tutorial.

The CWC calculation functions (calc_ewt and calc_ewt_neon) are expecting the data and the k_liquid_water_ice.csv file to be stored in a directory that is at the same file level as where you have this notebook stored. In the cell below, the `data_dir = r"../data"` code ensures that the data directory will be at the same file level as where this tutorial notebook is stored.

Here is a visual of how the directory and file structure will look once these directories are created and files are downloaded:

```
project-root/
│
├── data/                     # Main folder for data
│   ├── cwc/                  # Subfolder for canopy water content (CWC) data genearted in tutorial_notebook_02
│   │   ├── emit_burn_cwc.nc               
│   │   ├── emit_burn_cwc.tif               
│   │   ├── emit_unburn_cwc.nc
│   │   ├── emit_unburn_cwc.tif                           
│   │   ├── neon_burn_cwc.nc               
│   │   ├── neon_burn_cwc.tif               
│   │   ├── neon_unburn_cwc.nc               
│   │   └── neon_unburn_cwc.tif          
│   │
│   ├── refl/                 # Subfolder for reflectance data generated in tutorial_notebook_01
│   │   ├── EMIT_L2A_RFL_20230731.nc
│   │   ├── EMIT_L2A_RFL_20230731_SOAP.nc
│   │   ├── EMIT_L2A_RFL_20230731_SOAP_burned.nc
│   │   ├── EMIT_L2A_RFL_20230731_SOAP_unburned.nc
│   │   ├── neon_burn_refl_float32.nc
|   |   └── neon_unburn_refl_float32.nc
|   └── k_liquid_water_ice.csv
│
└── notebooks/                # Subfolder for modules and tutorial notebooks
    ├── modules/              # Subfolder for Python scripts for processing and analysis
    │   ├── .ipynb_checkpoints
    |   ├── __pycache__
    |   ├── __init__
    |   ├── ewt_tools.py
    |   ├── ewt_calc2.py
    |   └── test_functions.py
    ├── tutorial_notebook_01.ipynb
    └── tutorial_notebook_02.ipynb
```

In [ ]:
# Define the file path for the data directory
data_dir = r"../data"

# Create the data_dir if it doesn't already exist
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    print(f'data directory made here: {data_dir}')
else:
    print(f'data directory already exists here: {data_dir}')

In [ ]:
# List of directories names to make
dir_list = ["refl", "shapefiles"]

# Define the root path where the directories will be created
root_path = data_dir

# Use a for loop to create the directories in the dir_list
for dir_name in dir_list:
    full_path = os.path.join(root_path, dir_name)
    if not os.path.exists(full_path):
        os.makedirs(full_path)
        print(f'directory made here: {full_path}')
    else:
        print(f'directory already exists here: {full_path}')

In [ ]:
# Define the file path for the modules directory
modules_dir = r"../notebooks/modules"

# Create the data_dir if it doesn't already exist
if not os.path.exists(modules_dir):
    os.makedirs(modules_dir)
    print(f'modules directory made here: {modules_dir}')
else:
    print(f'modules directory already exists here: {modules_dir}')

### 1.2 Download and Import Necessary Scripts

In [ ]:
# Import functions from the python scripts in the modules directory
from modules.emit_tools import emit_xarray # open EMIT datasets into xarray.Dataset
from modules.test_functions import surfrfl_hvplot_image

If not already installed, install the neonutilities packages 
using pip as follows:
!pip install neonutilities
!pip install python-dotenv

For this notebook:
* NEON & EMIT co-located data are provided
* EMIT L2A Reflectance granule is downloaded using earthaccess using an URL

In [ ]:
import neonutilities as nu
import dotenv

### Download the SOAP Burned and Unburned Reflectance Tiles:
* Burned: <a href="https://storage.googleapis.com/neon-aop-provisional-products/2024/FullSite/D17/2024_SOAP_8/L3/Spectrometer/Reflectance/NEON_D17_SOAP_DP3_298000_4100000_bidirectional_reflectance.h5" class="link--button link--arrow">NEON_D17_SOAP_DP3_298000_4100000_bidirectional_reflectance.h5</a>

* Unburned: <a href="https://storage.googleapis.com/neon-aop-provisional-products/2024/FullSite/D17/2024_SOAP_8/L3/Spectrometer/Reflectance/NEON_D17_SOAP_DP3_298000_4101000_bidirectional_reflectance.h5" class="link--button link--arrow">NEON_D17_SOAP_DP3_298000_4101000_bidirectional_reflectance.h5</a>

Login to your NASA Earthdata account and 
create a .netrc file using the login function from the earthaccess library. 
If you do not have an Earthdata Account, you can create one here.

In [ ]:
earthaccess.login(persist=True)

In [ ]:
url = 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/EMITL2ARFL.001/EMIT_L2A_RFL_001_20230731T205320_2321214_004/EMIT_L2A_RFL_001_20230731T205320_2321214_004.nc'

Get an HTTPS Session using your earthdata login, set a local path to save the file, and download the granule asset - This may take a while, the reflectance file is approximately 1.8 GB.

In [ ]:
# Get requests https Session using Earthdata Login Info
fs = earthaccess.get_requests_https_session()
# Retrieve granule asset ID from URL (to maintain existing naming convention)
granule_asset_id = url.split('/')[-1]
# Define Local Filepath
fp = f'../data/REFL/{granule_asset_id}'
# Download the Granule Asset if it doesn't exist
if not os.path.isfile(fp):
    with fs.get(url,stream=True) as src:
        with open(fp,'wb') as dst:
            for chunk in src.iter_content(chunk_size=64*1024*1024):
                dst.write(chunk)

# 2.0 Data in NEON Region of Interest 
* Create a NEON API token following [this tutorial](https://www.neonscience.org/resources/learning-hub/tutorials/neon-api-tokens-tutorial).
* Load a shapefile of the NEON site boundaries and SOAP reflectance data.


In [ ]:
# dotenv.set_key(dotenv_path=".env",
# key_to_set="NEON_TOKEN",
# value_to_set="your-token-here") - Use NEON token generated from your account

In [ ]:
# function to download data stored on the internet in a public url to a local file
def download_url(url,data_dir):
    if not os.path.isdir(data_dir):
        os.makedirs(data_dir)
    filename = url.split('/')[-1]
    r = requests.get(url, allow_redirects=True)
    file_object = open(os.path.join(data_dir,filename),'wb')
    file_object.write(r.content)

In [ ]:
# Download and Unzip the NEON Flight Boundary Shapefile 
neon_boundary_url = "https://www.neonscience.org/sites/default/files/AOP_flightBoxes_0.zip"
# Use download_url function to save the file to a directory
os.makedirs('../data/shapefiles/', exist_ok=True)
download_url(neon_boundary_url,'../data/shapefiles/')
# Unzip the file
with ZipFile(f"../data/shapefiles/{neon_boundary_url.split('/')[-1]}", 'r') as zip_ref:
    zip_ref.extractall('../data/shapefiles/')

In [ ]:
aop_flightboxes = gpd.read_file("../data/shapefiles/AOP_flightBoxes/AOP_flightboxesAllSites.shp")
aop_flightboxes.head()

In [ ]:
site_id = 'SOAP'
aop_flightboxes[aop_flightboxes.siteID == site_id]

## 2.1 EMIT Data

EMIT L2A Reflectance Data are distributed in a non-orthocorrected spatially raw NetCDF4 (.nc) format consisting of the data and its associated metadata. To work with this data, we will use the emit_xarray function from the emit_tools.py module included in the repository.

In [ ]:
ds_nc = nc.Dataset(fp)
ds_nc

In [ ]:
ds_nc['location']

## 2.1.2 Crop EMIT Data to ROI

To make the rest of this code run quicker and to make the CWC calculation less intensive, crop the EMIT granule to the SOAP flight boxes now. Taken from [Hannah's 07 notebook](https://github.com/NEONScience/AOP-EMIT/blob/hrieder/notebooks/exploratory/hr/07_hr_cwc_emit.ipynb)

In [ ]:
#open a shapefile of the ROI
aop_flightboxes = gpd.read_file("../data/shapefiles/AOP_flightboxes")
soap_polygon = aop_flightboxes[aop_flightboxes.siteID == 'SOAP']
shape = soap_polygon
shape

In [ ]:
#define EMIT file path
emit_fp = ("../data"
           "/refl"
           "/EMIT_L2A_RFL_001_20230731T205320_2321214_004.nc")

In [ ]:
#learn about emit_xarray function
help(emit_xarray)

In [ ]:
#open emit_fp
emit_ds = emit_xarray(
    #filepath
    emit_fp,
    #orthorectify the dataset
    ortho=True
).load()

#check dataset
emit_ds

In [ ]:
#crop emit_ds to SOAP flightboxes
emit_crop_SOAP_ds = emit_ds.rio.clip(
    #crop to SOAP polygon geometry
    shape.geometry.values,
    #crop to SOAP polygon CRS
    shape.crs,
    #include all pixels touched by polygon
    all_touched=True)

In [ ]:
#export emit_crop_ds and save to filepath we can use in calc_ewt fxn
emit_crop_SOAP_ds.to_netcdf("../data/refl"
           "/EMIT_L2A_RFL_20230731_SOAP.nc")

#define filepath
emit_crop_SOAP_fp = ("../data/refl"
           "/EMIT_L2A_RFL_20230731_SOAP.nc")

#check filepath
emit_crop_SOAP_fp

In [ ]:
# open emit_crop_SOAP_fp using code from line 37 of ewt_calc.py in notebook 2
# we did this to see if this helps the final fxn run faster
# we also did this to open the cropped dataset in the same way that the 
# calc_ewt fxn does to see if that helps reduce errors.
emit_crop_SOAP_ds = xr.open_dataset(emit_crop_SOAP_fp, decode_coords="all")

# check dataset
emit_crop_SOAP_ds

In [ ]:
# check emit_crop_ds.reflectance after loading in w/ ewt_calc.py code
emit_crop_SOAP_ds.reflectance

In [ ]:
# check emit_crop_SOAP_ds.reflectance after loading in w/ ewt_calc.py code.
# wanting to check NaN values and reflectance values in general
# to make sure they're NaN values and not -3000000 from cropping and exporting above
emit_crop_SOAP_ds.reflectance.plot.hist()

In [ ]:
# view surface reflectance of cropped area for wavelength closest to 850
emit_crop_SOAP_ds.sel(
    wavelengths=850,
    #use nearest valid index value
    method='nearest').reflectance.plot()

In [ ]:
# open burned tile of interest shapefile

# define filepath for burned tiles
burned_tile_shp_fp = ('..\\data\\shapefiles'
                   '\\NEON_D17_SOAP_DPQA_298000_4100000_boundary.shp')

# write burned tile boundary filepath to geodataframe
burned_tile_gdf = gpd.read_file(burned_tile_shp_fp)

# check geodataframe
burned_tile_gdf

In [ ]:
# open unburned tile of interest shapefile

# define filepath for burned tiles
unburned_tile_shp_fp = ('..\\data\\shapefiles'
                   '\\NEON_D17_SOAP_DPQA_298000_4101000_boundary.shp')

# write burned tile boundary filepath to geodataframe
unburned_tile_gdf = gpd.read_file(unburned_tile_shp_fp)

# check geodataframe
unburned_tile_gdf

In [ ]:
# crop EMIT granule to burned tile of interest
EMIT_L2A_RFL_20230731_SOAP_burned_ds = emit_crop_SOAP_ds.rio.clip(
    #crop to burned tile polygon geometry
    burned_tile_gdf.geometry.values,
    #crop to burned tile polygon CRS
    burned_tile_gdf.crs,
    #include all pixels touched by polygon
    all_touched=True)
# check emit_burn_ds
EMIT_L2A_RFL_20230731_SOAP_burned_ds

In [ ]:
# crop EMIT granule to burned tile of interest
EMIT_L2A_RFL_20230731_SOAP_unburned_ds = emit_crop_SOAP_ds.rio.clip(
    #crop to burned tile polygon geometry
    unburned_tile_gdf.geometry.values,
    #crop to burned tile polygon CRS
    unburned_tile_gdf.crs,
    #include all pixels touched by polygon
    all_touched=True)
# check emit_burn_ds
EMIT_L2A_RFL_20230731_SOAP_unburned_ds

In [ ]:
# view surface reflectance of burned area for wavelength closest to 850
EMIT_L2A_RFL_20230731_SOAP_burned_ds.sel(
    wavelengths=850,
    #use nearest valid index value
    method='nearest').reflectance.plot()

In [ ]:
# view surface reflectance of burned area for wavelength closest to 850
EMIT_L2A_RFL_20230731_SOAP_unburned_ds.sel(
    wavelengths=850,
    #use nearest valid index value
    method='nearest').reflectance.plot()

In [ ]:
# export EMIT cropped to burned NEON tile area and save for calc_ewt fxn
EMIT_L2A_RFL_20230731_SOAP_burned_ds.to_netcdf("../data/refl/"
           "/EMIT_L2A_RFL_20230731_SOAP_burned.nc")

# define filepath
EMIT_L2A_RFL_20230731_SOAP_unburned_fp = ("../data/refl/"
           "EMIT_L2A_RFL_20230731_SOAP_burned.nc")

# check filepath
EMIT_L2A_RFL_20230731_SOAP_unburned_fp

In [ ]:
# export EMIT cropped to unburned NEON tile area and save for calc_ewt fxn
EMIT_L2A_RFL_20230731_SOAP_unburned_ds.to_netcdf("../data/refl/"
           "/EMIT_L2A_RFL_20230731_SOAP_unburned.nc")

# define filepath
EMIT_L2A_RFL_20230731_SOAP_unburned_fp = ("../data/refl/"
           "EMIT_L2A_RFL_20230731_SOAP_unburned.nc")

# check filepath
EMIT_L2A_RFL_20230731_SOAP_unburned_fp

## 2.2 NEON Data

In [ ]:
# Inspect NEON data to verify attributes

def inspect_h5_structure(file_path):
    """
    Opens an HDF5 file and prints its complete internal structure, including
    the names and shapes of all datasets.
    
    Args:
        file_path (str): The path to the HDF5 file to inspect.
    """
    if not os.path.exists(file_path):
        print(f"Error: The file '{file_path}' does not exist.")
        return

    try:
        with h5py.File(file_path, 'r') as hdf5_file:
            print(f"--- Inspecting HDF5 file: {file_path} ---")
            
            def recurse_h5_structure(group, indent=0):
                """Recursively prints the structure of a group."""
                for key in group.keys():
                    item = group[key]
                    if isinstance(item, h5py.Group):
                        print('  ' * indent + f"GROUP: {key}")
                        recurse_h5_structure(item, indent + 1)
                    elif isinstance(item, h5py.Dataset):
                        print('  ' * indent + f"DATASET: {key} (Shape: {item.shape})")
                        if 'Wavelength' in key or 'wavelength' in key:
                            print('  ' * (indent + 1) + "--> This dataset's name contains 'wavelength'. Check its shape.")
                        
            recurse_h5_structure(hdf5_file)
        
        print("\n--- Inspection Complete ---")
        print("Look for a DATASET named 'Wavelength' or similar with a shape of (1000,).")

    except Exception as e:
        print(f"An error occurred while inspecting the file: {e}")

# --- Example Usage ---
# The path to your NEON .h5 file.
h5_file_path = r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\FINAL\data\refl\NEON_D17_SOAP_DP3_298000_4100000_burned.h5"

inspect_h5_structure(h5_file_path)

In [ ]:
def aop_h5refl2xarray(h5_filename):
    """
    Reads a NEON AOP reflectance HDF5 file and returns an xarray.Dataset with reflectance and weather quality indicator data.

    Parameters
    ----------
    h5_filename : str
        Path to the NEON AOP reflectance HDF5 file.

    Returns
    -------
    dsT : xarray.Dataset
        An xarray Dataset containing:
            - 'reflectance': DataArray of reflectance values (y, x, wavelengths)
            - 'weather_quality_indicator': DataArray of weather quality indicator (y, x)
            - Coordinates: y (UTM northing), x (UTM easting), wavelengths, fwhm, good_wavelengths
            - Metadata attributes: projection, spatial_ref, EPSG, no_data_value, scale_factor, bad_band_window1, bad_band_window2, etc.
    """
    import h5py
    import numpy as np
    import xarray as xr

    with h5py.File(h5_filename) as hdf5_file:
        print('Reading in ', h5_filename)
        sitename = list(hdf5_file.keys())[0]
        h5_refl_group = hdf5_file[sitename]['Reflectance']
        refl_dataset = h5_refl_group['Reflectance_Data']
        refl_array = refl_dataset[()].astype('float32')

        # Transpose and flip reflectance data
        refl_arrayT = np.transpose(refl_array, (1, 0, 2))
        refl_arrayT = refl_array[::-1, :, :]

        refl_shape = refl_arrayT.shape
        wavelengths = h5_refl_group['Metadata']['Spectral_Data']['Wavelength'][:]
        fwhm = h5_refl_group['Metadata']['Spectral_Data']['FWHM'][:]

        # Weather Quality Indicator: transpose and flip to match reflectance
        wqi_array = h5_refl_group['Metadata']['Ancillary_Imagery']['Weather_Quality_Indicator'][()]
        wqi_arrayT = np.transpose(wqi_array, (1, 0))
        wqi_arrayT = wqi_array[::-1, :]

        # Collect metadata
        metadata = {}
        metadata['shape'] = refl_shape
        metadata['no_data_value'] = float(refl_dataset.attrs['Data_Ignore_Value'])
        metadata['scale_factor'] = float(refl_dataset.attrs['Scale_Factor'])
        metadata['bad_band_window1'] = h5_refl_group.attrs['Band_Window_1_Nanometers']
        metadata['bad_band_window2'] = h5_refl_group.attrs['Band_Window_2_Nanometers']
        metadata['projection'] = h5_refl_group['Metadata']['Coordinate_System']['Proj4'][()].decode('utf-8')
        metadata['spatial_ref'] = h5_refl_group['Metadata']['Coordinate_System']['Coordinate_System_String'][()].decode('utf-8')
        metadata['EPSG'] = int(h5_refl_group['Metadata']['Coordinate_System']['EPSG Code'][()])

        # Parse map info for georeferencing
        map_info = str(h5_refl_group['Metadata']['Coordinate_System']['Map_Info'][()]).split(",")
        pixel_width = float(map_info[5])
        pixel_height = float(map_info[6])
        x_min = float(map_info[3]); x_min = int(x_min)
        y_max = float(map_info[4]); y_max = int(y_max)
        x_max = x_min + (refl_shape[1]*pixel_width); x_max = int(x_max)
        y_min = y_max - (refl_shape[0]*pixel_height); y_min = int(y_min)

        # Calculate UTM coordinates for x and y axes
        x_coords = np.linspace(x_min, x_max, num=refl_shape[1]).astype(float)
        y_coordsT = np.linspace(y_min, y_max, num=refl_shape[0]).astype(float)

        # Flag good/bad wavelengths (1=good, 0=bad)
        good_wavelengths = np.ones_like(wavelengths)
        for bad_window in [metadata['bad_band_window1'], metadata['bad_band_window2']]:
            bad_indices = np.where((wavelengths >= bad_window[0]) & (wavelengths <= bad_window[1]))[0]
            good_wavelengths[bad_indices] = 0
        good_wavelengths[-10:] = 0

        # Create xarray DataArray for reflectance
        refl_xrT = xr.DataArray(
            refl_arrayT,
            dims=["y", "x", "wavelengths"],
            name="reflectance",
            coords={
                "y": ("y", y_coordsT),
                "x": ("x", x_coords),
                "wavelengths": ("wavelengths", wavelengths),
                "fwhm": ("wavelengths", fwhm),
                "good_wavelengths": ("wavelengths", good_wavelengths)
            }
        )

        # Create xarray DataArray for Weather Quality Indicator
        wqi_xrT = xr.DataArray(
            wqi_arrayT,
            dims=["y", "x"],
            name="weather_quality_indicator",
            coords={
                "y": ("y", y_coordsT),
                "x": ("x", x_coords)
            }
        )

        # Create xarray Dataset and add metadata as attributes
        dsT = xr.Dataset({
            "reflectance": refl_xrT,
            "weather_quality_indicator": wqi_xrT
        })
        for key, value in metadata.items():
            if key not in ['shape', 'extent', 'ext_dict']:
                dsT.attrs[key] = value

        return dsT

In [ ]:
# Verify the function works successfully 
burned_reflectance_da = aop_h5refl2xarray(r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\FINAL\data\refl\NEON_D17_SOAP_DP3_298000_4100000_burned.h5")

# Print the object
print(burned_reflectance_da)


In [ ]:
# Verify the function works successfully 
unburned_reflectance_da = aop_h5refl2xarray(r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\FINAL\data\refl\NEON_D17_SOAP_DP3_298000_4101000_unburned.h5")

# Print the object
print(unburned_reflectance_da)

In [ ]:
# Now convert to EMIT compatible format (.nc)
def convert_h5_to_nc_in_place(folder):
    """
    Converts all .h5 reflectance files in a folder to .nc files,
    saving the output in the same folder.

    Args:
        folder (str): The directory containing the input .h5 files.
    """
    # Use pathlib to work with the directory path
    folder_path = pathlib.Path(folder)
    
    # Check if the folder exists
    if not folder_path.is_dir():
        print(f"Error: The specified folder '{folder}' does not exist.")
        return

    # Iterate through all files in the directory
    for file_path in folder_path.iterdir():
        if file_path.suffix == ".h5":
            input_h5_path = file_path
            output_nc_path = file_path.with_suffix(".nc")
            
            # Check if the .nc file already exists to avoid reprocessing
            if output_nc_path.exists():
                print(f"Skipping {file_path.name}: .nc file already exists.")
                continue

            print(f"Converting {file_path.name}...")
            
            try:
                # Step 1: Use the specialized function to load the HDF5 file
                # The function returns None on error, so we need to check for it.
                reflectance_data = aop_h5refl2xarray(str(input_h5_path))
                
                if reflectance_data is not None:
                    # Step 2: Write the xarray object to a NetCDF file
                    reflectance_data.to_netcdf(str(output_nc_path))
                    print(f"Successfully converted and saved to {output_nc_path.name}")
                else:
                    print(f"Failed to convert {file_path.name}, skipping.")
                    
            except Exception as e:
                print(f"An error occurred while processing {file_path.name}: {e}")
            
    print("\nBatch conversion complete.")


# --- Example Usage ---
# Define your single folder for input and output
data_folder = r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\FINAL\data\refl"

# Run the conversion
convert_h5_to_nc_in_place(data_folder)

# 3.0 Visualize Data

In [ ]:
# plot burned dataset to check it was loaded back in correctly
surfrfl_hvplot_image(
    EMIT_L2A_RFL_20230731_SOAP_burned_ds.sel(
        wavelengths=850,
        # use nearest valid index value
        method='nearest'),
    plottitle='SOAP Burned Tile EMIT Surface Reflectance, 850.1 nm')

In [ ]:
# plot unburned dataset to check it was loaded back in correctly
surfrfl_hvplot_image(
    EMIT_L2A_RFL_20230731_SOAP_unburned_ds.sel(
        wavelengths=850,
        # use nearest valid index value
        method='nearest'),
    plottitle='SOAP Unburned Tile EMIT Surface Reflectance, 850.1 nm')

In [ ]:
# function to adjust NEON contrast for RGB images
def gamma_adjust(rgb_ds, bright=0.2, white_background=False):
    array = rgb_ds.reflectance.data
    gamma = math.log(bright)/math.log(np.nanmean(array)) # Create exponent for gamma scaling - can be adjusted by changing 0.2 
    scaled = np.power(np.nan_to_num(array,nan=1),np.nan_to_num(gamma,nan=1)).clip(0,1) # Apply scaling and clip to 0-1 range
    if white_background == True:
        scaled = np.nan_to_num(scaled, nan = 1) # Set NANs to 1 so they appear white in plots
    rgb_ds.reflectance.data = scaled
    return rgb_ds

# Plot the RGB image of the burned SOAP tile
neon_burn_rgb = burned_reflectance_da.sel(wavelengths=[650, 560, 470], method='nearest')
neon_burn_rgb = gamma_adjust(neon_burn_rgb,white_background=True)
neon_burn_rgb.hvplot.rgb(y='y',x='x',bands='wavelengths',
                         xlabel='UTM x',ylabel='UTM y',
                         title='NEON AOP Reflectance RGB - SOAP Burned Tile',
                         frame_width=480, frame_height=480)

In [ ]:
# function to adjust NEON contrast for RGB images
def gamma_adjust(rgb_ds, bright=0.2, white_background=False):
    array = rgb_ds.reflectance.data
    gamma = math.log(bright)/math.log(np.nanmean(array)) # Create exponent for gamma scaling - can be adjusted by changing 0.2 
    scaled = np.power(np.nan_to_num(array,nan=1),np.nan_to_num(gamma,nan=1)).clip(0,1) # Apply scaling and clip to 0-1 range
    if white_background == True:
        scaled = np.nan_to_num(scaled, nan = 1) # Set NANs to 1 so they appear white in plots
    rgb_ds.reflectance.data = scaled
    return rgb_ds

# Plot the RGB image of the burned SOAP tile
neon_unburn_rgb = unburned_reflectance_da.sel(wavelengths=[650, 560, 470], method='nearest')
neon_unburn_rgb = gamma_adjust(neon_unburn_rgb,white_background=True)
neon_unburn_rgb.hvplot.rgb(y='y',x='x',bands='wavelengths',
                         xlabel='UTM x',ylabel='UTM y',
                         title='NEON AOP Reflectance RGB - SOAP Unburned Tile',
                         frame_width=480, frame_height=480)

# Continue to Notebook 2 to the Analysis of Co-located Data

## 4.0 Finding Data for Other Locations
To find other locations where NEON and EMIT AOP data overlap, please see this 
notebook: 
* [https://github.com/NEONScience/AOP-EMIT/blob/main/notebooks/exploratory/bh/01_bh_find_collocated_neon_emit_data.ipynb](https://github.com/NEONScience/AOP-EMIT/blob/main/notebooks/exploratory/bh/01_bh_find_collocated_neon_emit_data.ipynb)

See these links for a complete listing of data products:
1. NEON: [https://data.neonscience.org/data-products/explore](https://data.neonscience.org/data-products/explore)
2. EMIT: [https://github.com/nasa/EMIT-Data-Resources](https://github.com/nasa/EMIT-Data-Resources)

NEON spatial data & maps can be found here: [https://www.neonscience.org/data-samples/data/spatial-data-maps](https://www.neonscience.org/data-samples/data/spatial-data-maps)